# 🧠 プロンプト学習 (Prompt Tuning) with PEFT

このNotebookでは、**軽量な日本語GPT-2モデル**を使ってプロンプト学習を体験します。

- モデル: `rinna/japanese-gpt2-small` (~120MB)
- 手法: Prompt Tuning（PEFTライブラリ使用）
- タスク: テキスト生成（感情分析プロンプトの学習）

**💡 ポイント**: モデルの重みは一切変更せず、わずかなソフトトークンだけを学習します！

---
**実行環境**: Google Colab (T4 GPU推奨)

## ① ライブラリのインストール

In [ ]:
!pip install -q peft transformers datasets sentencepiece accelerate
print('✅ インストール完了')

## ② GPU確認

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')

if device == 'cuda':
    print(f'GPU名: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('⚠️ GPUが見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選択してください')

## ③ モデルとトークナイザーの読み込み

`rinna/japanese-gpt2-small` は約120MBの軽量な日本語GPT-2モデルです。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'rinna/japanese-gpt2-small'

print('モデルを読み込み中...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)

# パラメータ数を確認
total_params = sum(p.numel() for p in model.parameters())
print(f'\n✅ モデル読み込み完了')
print(f'総パラメータ数: {total_params:,} ({total_params/1e6:.1f}M)')

## ④ プロンプト学習前のテキスト生成（ベースライン）

プロンプト学習を行う前に、モデルの生成を確認します。

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=50):
    """テキストを生成するヘルパー関数"""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated

# ベースライン生成
test_prompt = 'この映画はとても'
print(f'入力: {test_prompt}')
print(f'生成: {generate_text(model, tokenizer, test_prompt)}')

## ⑤ PEFTでプロンプトチューニングの設定

- `num_virtual_tokens`: 学習するソフトトークン数（多いほど表現力が上がるが学習が重い）
- `prompt_tuning_init_text`: ソフトトークンの初期化テキスト

In [ ]:
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model, TaskType

# プロンプトチューニングの設定
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=10,                          # ← 学習するソフトトークン数
    prompt_tuning_init_text='映画のレビューを書いてください:',  # ← 初期化テキスト
    tokenizer_name_or_path=MODEL_NAME,
)

# PEFTモデルに変換
peft_model = get_peft_model(model, peft_config)

# 学習可能パラメータを確認
peft_model.print_trainable_parameters()

## ⑥ 学習データの準備

シンプルな映画レビューデータを作成します。

In [ ]:
from torch.utils.data import Dataset, DataLoader

# 学習用データ（映画レビュー）
train_texts = [
    'この映画はとても感動的で、涙が止まりませんでした。',
    'ストーリーが素晴らしく、最後まで目が離せませんでした。',
    'キャストの演技が圧巻で、何度でも見たい作品です。',
    'この映画は期待外れで、途中で飽きてしまいました。',
    '映像美は素晴らしいですが、脚本が弱いのが残念です。',
    '家族で楽しめる、心温まる素敵な映画でした。',
    '主人公の成長が丁寧に描かれていて、共感できました。',
    'アクションシーンが迫力満点で、大興奮しました。',
]

class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=64):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['input_ids'][idx].clone(),
        }

dataset = ReviewDataset(train_texts, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f'✅ データセット準備完了: {len(dataset)}件')

## ⑦ 学習ループ

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

# オプティマイザ（学習可能パラメータのみ）
optimizer = AdamW(peft_model.parameters(), lr=3e-3)

NUM_EPOCHS = 10
loss_history = []

peft_model.train()
print('🚀 学習開始...')

for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # パディングトークンのlossを無視
        labels[labels == tokenizer.pad_token_id] = -100

        outputs = peft_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    loss_history.append(avg_loss)
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS}  Loss: {avg_loss:.4f}')

print('\n✅ 学習完了！')

## ⑧ 学習曲線の可視化

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'IPAexGothic' if 'IPAexGothic' in matplotlib.font_manager.findSystemFonts() else 'DejaVu Sans'

plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), loss_history, marker='o', color='royalblue', linewidth=2)
plt.title('学習曲線 (Loss)', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'初期Loss: {loss_history[0]:.4f} → 最終Loss: {loss_history[-1]:.4f}')

## ⑨ 学習後のテキスト生成

プロンプト学習後、映画レビュースタイルの文章が生成されるか確認します。

In [ ]:
peft_model.eval()

test_prompts = [
    'この映画はとても',
    'ストーリーが',
    'キャストの演技が',
]

print('📝 学習後のテキスト生成:\n')
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f'入力: {prompt}')
    print(f'生成: {generated}')
    print('-' * 50)

## ⑩ モデルの保存と読み込み

学習したソフトトークン（アダプター）だけを保存します。サイズが非常に小さいのが特徴です！

In [ ]:
import os

SAVE_PATH = '/content/prompt_tuning_adapter'

# アダプターのみ保存（ソフトトークンだけ）
peft_model.save_pretrained(SAVE_PATH)

# ファイルサイズを確認
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f))
    print(f'{f}: {size/1024:.1f} KB')

print('\n✅ 保存完了！（ソフトトークンのみ、非常にコンパクト）')

In [ ]:
from peft import PeftModel

# ベースモデルを再読み込みしてアダプターを適用
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
loaded_model = PeftModel.from_pretrained(base_model, SAVE_PATH)
loaded_model.eval()

print('✅ アダプター読み込み完了！')

# 読み込み後の生成テスト
prompt = 'この映画はとても'
inputs = tokenizer(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    out = loaded_model.generate(
        **inputs, max_new_tokens=40,
        do_sample=True, temperature=0.8, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
print(f'生成: {tokenizer.decode(out[0], skip_special_tokens=True)}')

## ⑪ Google Driveへの保存（オプション）

セッションが終わってもアダプターを保持したい場合はGoogle Driveに保存します。

In [ ]:
# Google Driveのマウント（実行すると認証が求められます）
# from google.colab import drive
# drive.mount('/content/drive')

# Driveへコピー
# import shutil
# shutil.copytree(SAVE_PATH, '/content/drive/MyDrive/prompt_tuning_adapter')
# print('✅ Google Driveに保存しました')

print('💡 上のコメントを外して実行するとGoogle Driveに保存できます')

---

## 📊 まとめ

| 項目 | 内容 |
|------|------|
| 使用モデル | `rinna/japanese-gpt2-small` (~117M params) |
| 学習手法 | Prompt Tuning (PEFT) |
| 学習パラメータ | ソフトトークン10個のみ（全体の0.01%未満） |
| タスク | 映画レビュースタイルの文章生成 |
| GPU | T4 (15GB VRAM) で十分動作 |

### 🔧 カスタマイズのヒント

- `num_virtual_tokens` を増やす → 表現力UP（5〜20が目安）
- `train_texts` を変える → 別のタスクに適用可能
- `NUM_EPOCHS` を増やす → より収束（過学習に注意）
- `lr` を調整 → 3e-3〜1e-2 が Prompt Tuning では一般的

### 🚀 次のステップ

- より大きなモデル（`cyberagent/open-calm-3b`）に挑戦
- **Prefix Tuning** や **P-Tuning v2** を試す
- 分類タスク（ポジティブ/ネガティブ判定）に応用する